<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Synthetic control

**[Synthetic control](https://learning.nextia-ai.com/courses/experiments/m05/synthetic-control/)** · Experiments and Causal Impact: Did Our Change Work? · Module 5, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build a synthetic Quebec from a weighted mix of the other provinces, use it as Quebec's counterfactual after free delivery started, and test the result with a placebo test for every other province.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | difference-in-differences, from [Difference-in-differences](https://learning.nextia-ai.com/courses/experiments/m05/difference-in-differences/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m05/synthetic-control/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M05-L02-synthetic-control/synthetic-control-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 "scipy==1.15.3; python_version < '3.14'" "scipy==1.18.1; python_version >= '3.14'" statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 2 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "regions/provinces_weekly.csv": "2e86c748e3f53a799e864e1949f8541f7feefedf1be0ec26d79751340ba71238",
    "regions/truth.json": "c1e0f29839d7451fb97d553553056a28e78c22aed5cf7e9f927ebbd559aada08"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A mix of two towns, by hand

Town A gets a new café in week 5. Neither town B nor town C moves like town A, but a mix of 60% B and 40% C does. The numbers are made up.

> **Predict.** What is the café's effect in week 5?

In [ ]:
a = np.array([100, 104, 108, 112, 126])
b = np.array([90, 92, 94, 96, 98])
c = np.array([115, 122, 129, 136, 143])
mix = 0.6 * b + 0.4 * c
print("mix:", mix.round(1))
print(f"effect in week 5: {a[4] - mix[4]:+.0f}")

> **Check.** You should see the mix `[100. 104. 108. 112. 116.]` and `effect in week 5: +10`.

## 2. Find the weights for Quebec

The computer tries weights that are zero or more and add up to 1, and keeps those with the smallest squared gap to Quebec in the weeks before 2 June 2025. As in the project, each province is taken on the log scale and relative to its own average before the change, so the mix matches the **shape** of Quebec's sales, not their size.

In [ ]:
from scipy.optimize import minimize

prov = pd.read_csv("data/regions/provinces_weekly.csv")
wide = prov.pivot(index="week", columns="province", values="revenue").sort_index()
pre = np.asarray(wide.index < "2025-06-02")
series = {p: np.log(wide[p].to_numpy()) for p in wide.columns}

def synth(series, treated, pre):
    """The weights, the gap (treated - mix, log scale) and the fit before the change."""
    donors = [u for u in series if u != treated]
    y = series[treated]
    x = np.column_stack([series[u] for u in donors])
    x_mean, y_mean = x[pre].mean(axis=0), y[pre].mean()
    loss = lambda w: float((((y[pre] - y_mean) - (x[pre] - x_mean) @ w) ** 2).sum())
    k = len(donors)
    res = minimize(loss, np.full(k, 1 / k), method="SLSQP", bounds=[(0, 1)] * k,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}], options={"maxiter": 500, "ftol": 1e-12})
    w = np.clip(res.x, 0, None)
    w = w / w.sum()
    gap = y - ((x - x_mean) @ w + y_mean)
    return dict(zip(donors, w)), gap

weights, gap = synth(series, "QC", pre)
for p, w in sorted(weights.items(), key=lambda kv: -kv[1]):
    print(f"{p} {w:.2f}")

> **Check.** You should see `BC 0.38` first, then `NL 0.13`, `SK 0.13`, `AB 0.12`, and `NS 0.00` last.

## 3. Quebec against the synthetic Quebec

In [ ]:
import matplotlib.pyplot as plt

qc = wide["QC"].to_numpy()
synthetic = np.exp(np.log(qc) - gap)
weeks = pd.to_datetime(wide.index)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(weeks, qc / 1000, label="Quebec")
ax.plot(weeks, synthetic / 1000, "--", label="synthetic Quebec")
ax.axvline(pd.Timestamp("2025-06-02"), color="grey", linestyle=":")
ax.set_ylabel("weekly sales (thousand dollars)")
ax.legend()
plt.show()

rmspe_pre = np.sqrt((gap[pre] ** 2).mean())
effect = np.expm1(gap[~pre].mean())
print(f"fit before: RMSPE {rmspe_pre:.4f}; effect {effect:+.1%}")
print(f"true effect (simulator): {truth('regions')['effect']['relative']:+.1%}")

> **Check.** You should see `fit before: RMSPE 0.0128; effect +5.5%` and `true effect (simulator): +6.0%`.

Before the change, the dashed line follows Quebec closely. After it, Quebec runs above the mix.

## 4. Your turn: a placebo test for every province

Pretend that each of the nine other provinces got free delivery on 2 June. Build its synthetic version from the other eight (Quebec is left out, because it really got the change). For each province, and for Quebec, compute the **ratio**: the RMSPE after the change divided by the RMSPE before.

> **Your turn.** Fill in `ratio`. Then find Quebec's `rank` among the 10 provinces (1 = the largest ratio).

In [ ]:
def ratio(series, unit):
    """RMSPE after the change / RMSPE before, for one unit and its synthetic version."""
    ...

donors = [p for p in series if p != "QC"]
rank = ...  # Quebec's place among the 10 ratios, 1 = largest

In [ ]:
expect("Quebec's rank", rank, 1)

Quebec ranks first, with a ratio of 4.4. With 10 provinces, the placebo p-value is 1 ÷ 10 = 0.10, the smallest possible. Over 50 seeds of this scenario, the course measured a mean synthetic-control estimate of +5.6% against the true +6.0%: the closest method in this module.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m05/synthetic-control/) has the placebo chart and the knowledge checks that count toward your certificate. In the project: `xp synth`, then `xp check`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Regression discontinuity](https://learning.nextia-ai.com/courses/experiments/m05/regression-discontinuity/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m05/synthetic-control/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.